# ML-02 â€” Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terddyy/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter dataset is built for exactly this decision — it ships `trend_direction`, `days_since_last_update`, `avg_position`, and `impressions_90d` side by side, which is what a scoring/queue approach needs. It's also the lane with the most runnable scaffolding already in this repo (`scripts/01`-`05`, the baseline + model pipeline, `outputs/model_report.md`), so I can validate my own reasoning against a known-good reference instead of building blind. Ranking Signal Analysis (Lane 1) and CTR Opportunity Scoring (Lane 4) are close seconds, but a review queue is the most direct route to a decision someone can act on this week, not just a report.

In [1]:
# Lane choice is qualitative (Section 1). Real supporting numbers land in Section 3.
print("Provisional lane: Refresh / Content Opportunity Scoring (Lane 2)")

Provisional lane: Refresh / Content Opportunity Scoring (Lane 2)


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a content item's trailing-90-day search and engagement signals, which pages should a content reviewer look at first this week?

**Unit of analysis:** one content item (`content_id`) — a single page, using its trailing-90-day metrics as of the snapshot date. Not a client, not a day.

**Output:** a ranked review queue — a score per page plus a reason code (e.g. `stale_visible_page`, `declining_with_demand`, `page_one_decay_risk`) explaining why it surfaced.

**Who acts, and what they do:** a content editor or SEO reviewer with limited weekly capacity (say, 20-50 pages) opens the queue and decides, page by page, whether to refresh, expand, or leave a page alone. The model doesn't take action — it orders their attention.

**Cost of a wrong call:**
- *False positive* (flagged high-priority, actually fine): a reviewer spends 30-60 minutes auditing a page that didn't need it — wasted editor time, opportunity cost against a page that did need it.
- *False negative* (missed, should have been flagged): a genuinely declining, high-demand page keeps losing visibility for another review cycle (commonly ~1-4 weeks) before anyone notices — lost traffic and, downstream, lost conversions.
Because editor time is the scarce resource and a miss just delays (rather than destroys) the fix, I'll weight precision at the top of the queue (precision@K) over raw recall, but I will report both.

**Why data or ML helps here:** no single column decides priority — a page can be low on trend, high on demand, thin on content, or stuck at position 8, and these signals interact and trade off against each other in ways that are hard to hand-write as one if/else chain covering 30,000 pages across 32 clients. A simple weighted rule is a reasonable *baseline* (and I'll build one), but I expect a learned ranking to separate real opportunity from noise better than any single threshold I could pick by eye — that's the comparison this project exists to make honest.

In [2]:
# Section 2 is framing, not a computed number — the check for cost/decision reasoning happens
# against real evidence in Section 3.
print("Decision: which pages to review first. Actor: content/SEO reviewer. Metric focus: precision@K.")

Decision: which pages to review first. Actor: content/SEO reviewer. Metric focus: precision@K.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV and checking whether this lane has enough real candidates to be worth building on for the next 7 weeks.

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
n = len(df)
print(f"Rows: {n:,} | Clients: {df['client_id'].nunique()}")

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
page_one_decay = ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).sum()
median_impressions = df["impressions_90d"].median()

print(f"declining_with_demand candidates: {declining_with_demand:,} ({declining_with_demand/n:.1%} of rows)")
print(f"page_one_decay_risk candidates: {page_one_decay:,} ({page_one_decay/n:.1%} of rows)")
print(f"median impressions_90d: {median_impressions:,.0f}")

Rows: 30,000 | Clients: 32
declining_with_demand candidates: 13,152 (43.8% of rows)
page_one_decay_risk candidates: 7,076 (23.6% of rows)
median impressions_90d: 731


**What these numbers say:** out of 30,000 pages across 32 clients, about 43.8% (13,152 pages) are already tagged `trend_direction == "down"` while still pulling meaningful demand (impressions_90d >= 100), and about 23.6% (7,076 pages) sit on page one (avg_position 1-10) but are old enough (180+ days) to be at decay risk. The median page in this dataset gets 731 impressions over 90 days — real, non-trivial traffic, not scraps. That's thousands of live candidates with demand behind them, which is exactly the volume a reviewer needs a *ranked* queue for rather than an eyeballed list — one more reason a review-first ranking (not a magic-fix guarantee) is the right shape for this lane.

## 4. Careful words: what I can and can't claim

**What this work can say:**
- *Observed*: these pages showed these trailing-90-day search/engagement patterns as of the snapshot.
- *Directional*: pages with these patterns (declining trend + demand, page-one decay, thinness) are associated with higher review priority in this dataset.
- *Decision-support*: a ranked queue that helps a reviewer spend limited time on the most promising candidates first, backed by reason codes they can inspect and override.

**What this work will never say:**
- That refreshing a flagged page *caused* a recovery — that requires a controlled experiment (e.g. before/after with a control group), which this observational dataset doesn't provide.
- That any signal here reflects a specific Google ranking factor — `trend_direction`, `avg_position`, and `ctr` are FlyRank's own measurements, not Google's internals.
- That the model output is a guarantee — it's a priority order under limited capacity, not a promise of results.
- Anything about individual clients, URLs, or queries — all IDs are pseudonymized and I'll only report aggregates.

In [4]:
print("Claims: observed + directional + decision-support only.")
print("Never: causal recovery claims, Google-algorithm claims, or client/URL identification.")

Claims: observed + directional + decision-support only.
Never: causal recovery claims, Google-algorithm claims, or client/URL identification.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled â€” markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.